# 08 — Monitoreo de Drift con Evidently AI

In [ ]:
!pip install evidently pandas numpy plotly scikit-learn -q

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("Imports OK")

## 1. El Problema del Model Decay

Un modelo de ML entrenado hoy no necesariamente funcionará bien en 6 meses.
El mundo cambia, y los modelos que no se actualizan empiezan a degradarse.

**¿Por qué se degrada un modelo?**

**Concept Drift** — la relación entre features y target cambia:
- Un modelo de demanda eléctrica entrenado en invierno asume patrones de calefacción.
  En verano, el patrón dominante es refrigeración: mismos inputs, diferente comportamiento.
- Un modelo de demanda entrenado antes del COVID no anticipó el cambio en los patrones de consumo.
- Un modelo de precios entrenado en 2019 no captura la inflación de 2022.

**Data Drift** — la distribución de los inputs cambia:
- Un sensor de temperatura falla y empieza a reportar valores constantes de 0°C.
- La demografía de los usuarios cambia (expansion a nuevos mercados).
- Un campo que antes era obligatorio ahora tiene 40% de nulos.

**Ejemplo concreto:**
Modelo de demanda eléctrica entrenado con datos de enero-junio (temperatura media 22°C).
Desplegado en producción en julio, cuando la temperatura sube a 35°C por olas de calor.
El modelo nunca vio temperaturas tan altas → predicciones muy por debajo de la demanda real
→ apagones por subestimación de capacidad necesaria.

**La solución**: monitoreo continuo + reentrenamiento automático cuando se detecta drift.

## 2. Data Drift vs Concept Drift

| Tipo | Definición | Detección | Ejemplo |
|------|-----------|-----------|--------|
| **Data Drift** | La distribución de P(X) cambia | Comparar distribuciones de features entre ref y actual | Temperatura media sube 13°C |
| **Concept Drift** | La relación P(Y\|X) cambia | Comparar predicciones vs actuals (requiere labels) | Nuevo patrón de consumo post-pandemia |
| **Label Drift** | La distribución de P(Y) cambia | Comparar distribución del target | Demanda media sube 35% |
| **Covariate Shift** | P(X) cambia pero P(Y\|X) igual | PSI, KS test en features | Nuevo sensor, misma física |

**En la práctica:**
- El data drift es más fácil de detectar (no necesita labels reales).
- El concept drift requiere que el sistema reciba feedback (labels) con cierta latencia.
- En producción, es común monitorear data drift como proxy del concept drift.

**Herramientas populares:**
- **Evidently AI**: librería Python open-source, genera reportes HTML. Usada en este notebook.
- **WhyLabs**: plataforma SaaS con integración a MLflow.
- **Arize AI**: plataforma enterprise con embeddings y LLM monitoring.
- **NannyML**: especializada en concept drift sin labels (CBPE).

## 3. Generar datos de referencia y actuales

In [ ]:
# Datos de referencia (periodo de entrenamiento: ene-jun)
rng = np.random.default_rng(42)
n_ref = 1000

reference_df = pd.DataFrame({
    "demand_kwh": rng.normal(1000, 80, n_ref),
    "temperature_c": rng.normal(22, 4, n_ref),   # temperatura de invierno/primavera
    "hour": rng.integers(0, 24, n_ref),
    "day_of_week": rng.integers(0, 7, n_ref),
    "demand_lag_24h": rng.normal(990, 85, n_ref),
})

# Datos actuales (periodo de producción: jul-dic) - SIN DRIFT
n_cur = 300
current_nodrift = pd.DataFrame({
    "demand_kwh": rng.normal(1010, 82, n_cur),     # distribución similar
    "temperature_c": rng.normal(23, 4.2, n_cur),
    "hour": rng.integers(0, 24, n_cur),
    "day_of_week": rng.integers(0, 7, n_cur),
    "demand_lag_24h": rng.normal(1000, 86, n_cur),
})

# Datos actuales CON DRIFT (verano con anomalía)
current_drift = pd.DataFrame({
    "demand_kwh": rng.normal(1350, 120, n_cur),    # +35% por calor
    "temperature_c": rng.normal(35, 3, n_cur),      # temperatura de verano
    "hour": rng.integers(0, 24, n_cur),
    "day_of_week": rng.integers(0, 7, n_cur),
    "demand_lag_24h": rng.normal(1330, 125, n_cur),
})

print(f"Referencia: {len(reference_df):,} registros | demand media: {reference_df['demand_kwh'].mean():.1f}")
print(f"Sin drift:  {len(current_nodrift):,} registros | demand media: {current_nodrift['demand_kwh'].mean():.1f}")
print(f"Con drift:  {len(current_drift):,} registros | demand media: {current_drift['demand_kwh'].mean():.1f}")

## 3. Simulación de Drift — Visualización de distribuciones

In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=reference_df["demand_kwh"], name="Referencia (entrenamiento)",
                            opacity=0.7, nbinsx=30))
fig.add_trace(go.Histogram(x=current_nodrift["demand_kwh"], name="Actual (sin drift)",
                            opacity=0.7, nbinsx=30))
fig.add_trace(go.Histogram(x=current_drift["demand_kwh"], name="Actual (con drift)",
                            opacity=0.7, nbinsx=30))
fig.update_layout(barmode="overlay", title="Distribución de demand_kwh: Referencia vs Actual",
                  xaxis_title="demand_kwh", yaxis_title="Frecuencia")
fig.show()

## 4. PSI: Population Stability Index

El PSI es la métrica más usada en la industria para detectar data drift.
Originalmente desarrollado en banca para monitorear scorecard de crédito.

**Fórmula:**
```
PSI = sum((cur_pct_i - ref_pct_i) * ln(cur_pct_i / ref_pct_i))
```

Donde:
- `ref_pct_i`: proporción de registros en el bucket i en los datos de referencia.
- `cur_pct_i`: proporción de registros en el bucket i en los datos actuales.

**Interpretación:**
| PSI | Significado | Acción |
|-----|-------------|--------|
| < 0.1 | Sin drift — distribuciones similares | Ninguna |
| 0.1 — 0.2 | Drift moderado — monitorear de cerca | Investigar |
| > 0.2 | Drift severo — reentrenar el modelo | Reentrenar |

**Ventajas del PSI:**
- Interpretación intuitiva y umbrales establecidos.
- No requiere distribución paramétrica.
- Funciona bien con variables continuas y discretas.

**Alternativas:**
- **KS Test (Kolmogorov-Smirnov)**: test estadístico de 2 muestras, más riguroso.
- **Wasserstein distance**: Earth Mover's Distance, sensible a cambios en la forma.
- **Jensen-Shannon divergence**: simétrica, entre 0 y ln(2).

In [ ]:
def compute_psi(reference, current, buckets=10):
    """PSI mide cuánto cambió la distribución de una variable."""
    min_val = min(reference.min(), current.min())
    max_val = max(reference.max(), current.max())
    bins = np.linspace(min_val, max_val, buckets + 1)

    ref_counts = np.histogram(reference, bins=bins)[0]
    cur_counts = np.histogram(current, bins=bins)[0]

    ref_pct = (ref_counts + 1e-6) / len(reference)
    cur_pct = (cur_counts + 1e-6) / len(current)

    psi = np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct))
    return float(psi)

# PSI por feature
features = ["demand_kwh", "temperature_c", "demand_lag_24h"]

print("=== PSI Sin Drift ===")
for feat in features:
    psi = compute_psi(reference_df[feat].values, current_nodrift[feat].values)
    status = "OK" if psi < 0.1 else ("MONITOREAR" if psi < 0.2 else "DRIFT!")
    print(f"  {feat:<25} PSI={psi:.4f}  [{status}]")

print("\n=== PSI Con Drift ===")
for feat in features:
    psi = compute_psi(reference_df[feat].values, current_drift[feat].values)
    status = "OK" if psi < 0.1 else ("MONITOREAR" if psi < 0.2 else "DRIFT!")
    print(f"  {feat:<25} PSI={psi:.4f}  [{status}]")

print("\nRegla de interpretación:")
print("  PSI < 0.1:  sin drift (distribuciones similares)")
print("  0.1 - 0.2:  drift moderado (monitorear de cerca)")
print("  PSI > 0.2:  drift severo (reentrenar el modelo)")

## 5. Evidently AI: Reporte de Drift

Evidently AI es una librería open-source que genera reportes automáticos de calidad de datos,
data drift y performance del modelo.

**Funcionalidades principales:**
- `DataDriftPreset`: detecta drift en todas las features automáticamente.
- `DataQualityPreset`: detecta nulos, duplicados, valores fuera de rango.
- `RegressionPreset` / `ClassificationPreset`: métricas de performance del modelo.
- Reportes HTML interactivos para dashboards.
- Integración con MLflow para loguear métricas de drift como experimentos.

In [ ]:
try:
    from evidently.report import Report
    from evidently.metric_preset import DataDriftPreset
    from evidently import ColumnMapping

    report = Report(metrics=[DataDriftPreset()])
    cm = ColumnMapping(numerical_features=features)

    report.run(
        reference_data=reference_df[features],
        current_data=current_drift[features],
        column_mapping=cm,
    )

    # Mostrar en notebook
    report.show(mode="inline")

    # Extraer resultados
    result = report.as_dict()
    print("Reporte generado con Evidently AI")

except ImportError:
    print("Evidently no instalado. Mostrando resultado del PSI manual:")
    for feat in features:
        psi = compute_psi(reference_df[feat].values, current_drift[feat].values)
        print(f"  {feat}: PSI={psi:.4f} → {'DRIFT' if psi > 0.2 else 'OK'}")
except Exception as e:
    print(f"Error con Evidently: {e}")
    print("Usando PSI manual como fallback...")
    for feat in features:
        psi = compute_psi(reference_df[feat].values, current_drift[feat].values)
        print(f"  {feat}: PSI={psi:.4f}")

## 6. Visualización de PSI por feature: Sin Drift vs Con Drift

In [ ]:
psi_scores_ref = {feat: compute_psi(reference_df[feat].values, current_nodrift[feat].values) for feat in features}
psi_scores_drift = {feat: compute_psi(reference_df[feat].values, current_drift[feat].values) for feat in features}

plot_df = pd.DataFrame({
    "feature": features * 2,
    "psi": list(psi_scores_ref.values()) + list(psi_scores_drift.values()),
    "scenario": ["Sin drift"] * len(features) + ["Con drift"] * len(features),
})

fig = px.bar(plot_df, x="feature", y="psi", color="scenario", barmode="group",
             title="PSI por feature: Sin Drift vs Con Drift",
             labels={"psi": "PSI Score", "feature": "Feature"},
             color_discrete_map={"Sin drift": "steelblue", "Con drift": "tomato"})
fig.add_hline(y=0.2, line_dash="dash", line_color="red", annotation_text="Threshold (0.2)")
fig.add_hline(y=0.1, line_dash="dash", line_color="orange", annotation_text="Alerta (0.1)")
fig.show()

## 6. Integrar con Airflow: disparar reentrenamiento

En el repositorio existe el DAG `dag_monitoring.py` que corre diariamente y:
1. Descarga los datos de producción del último período.
2. Calcula el PSI de todas las features vs el dataset de referencia.
3. Si PSI > 0.2 en alguna feature → dispara `dag_mlops_full_pipeline` via Airflow REST API.

```python
# Fragmento de dag_monitoring.py
def check_drift_and_trigger(**context):
    psi_scores = compute_all_psi(reference_data, current_data)
    max_psi = max(psi_scores.values())

    if max_psi > 0.2:
        # Disparar reentrenamiento via REST API de Airflow
        requests.post(
            f"{AIRFLOW_URL}/api/v1/dags/mlops_full_pipeline/dagRuns",
            auth=(AIRFLOW_USER, AIRFLOW_PASS),
            json={"conf": {"trigger_reason": "drift_detected", "max_psi": max_psi}},
        )
        print(f"Reentrenamiento disparado! Max PSI: {max_psi:.3f}")
    else:
        print(f"Sin drift. Max PSI: {max_psi:.3f} — modelo OK")
```

Este patrón permite **reentrenamiento automático basado en datos**, no en calendario.
El modelo se actualiza cuando lo necesita, no cada lunes a las 2AM independientemente.

## 7. Estrategias de Reentrenamiento

### A) Basado en tiempo (Time-based)
- Reentrenar cada N días (semanal, mensual).
- **Pros**: simple, predecible, fácil de implementar.
- **Contras**: puede ser demasiado frecuente o demasiado raro según el drift real.
- **Cuándo**: cuando los datos cambian a ritmo constante (estacionalidad clara).

### B) Basado en performance (Performance-based)
- Reentrenar cuando las métricas caen por debajo de un umbral (RMSE > X, F1 < Y).
- **Pros**: reacciona a degradación real del modelo.
- **Contras**: requiere labels con latencia aceptable (no funciona si el feedback tarda meses).
- **Cuándo**: cuando se dispone de ground truth con baja latencia.

### C) Basado en drift (Drift-based)
- Reentrenar cuando PSI > umbral en features críticas (como en `dag_monitoring.py`).
- **Pros**: proactivo — detecta el problema antes de que afecte las predicciones.
- **Contras**: puede generar falsos positivos; no todo drift causa degradación del modelo.
- **Cuándo**: cuando el feature drift es buen proxy del concept drift.

### D) Reentrenamiento completo vs incremental
| Tipo | Descripción | Cuándo usar |
|------|-------------|-------------|
| **Full retrain** | Reentrenar con todos los datos históricos + nuevos | Si el modelo necesita ver toda la historia |
| **Rolling window** | Reentrenar con ventana deslizante (ej: últimos 6 meses) | Si los datos históricos muy viejos dañan el modelo |
| **Incremental / Online** | Actualizar el modelo con nuevos batches sin reentrenar todo | Si el modelo soporta `partial_fit` (sklearn) o SGD |
| **Transfer learning** | Fine-tuning del modelo existente con nuevos datos | Modelos de deep learning, NLP |

**Recomendación práctica**: empezar con time-based (semanal/mensual), luego añadir
monitoreo de drift para detectar cambios abruptos entre reentrenamientos.

## Resumen

En este notebook aprendimos:

1. **Model decay**: los modelos se degradan porque el mundo cambia — es inevitable, no un bug.
2. **Data drift**: la distribución de P(X) cambia. Detectable sin labels.
3. **Concept drift**: la relación P(Y|X) cambia. Requiere ground truth para detectar.
4. **PSI**: métrica estándar de la industria. PSI > 0.2 = reentrenar. Implementación manual incluida.
5. **Evidently AI**: genera reportes completos de drift con una sola llamada.
6. **Integración con Airflow**: el DAG de monitoreo dispara reentrenamiento automáticamente cuando hay drift.
7. **Estrategias de reentrenamiento**: time-based, performance-based, drift-based; full vs incremental.

**El ciclo MLOps completo:**
```
Datos → Feature Eng → HPO → Entrenamiento → Evaluación → Registry
   ↑                                                          |
   |              Monitoreo de Drift                          |
   └──────────── Reentrenamiento automático ←── Producción ←──┘
```

Con este notebook termina el módulo de MLOps end-to-end. El pipeline completo
cubre desde la exploración de datos hasta el monitoreo en producción con reentrenamiento automático.